***

*Course:* [Math 535](https://people.math.wisc.edu/~roch/mmids/) - Mathematical Methods in Data Science (MMiDS)  
*Chapter:* 2-Least squares   
*Author:* [Sebastien Roch](https://people.math.wisc.edu/~roch/), Department of Mathematics, University of Wisconsin-Madison  
*Updated:* Feb 14, 2024   
*Copyright:* &copy; 2024 Sebastien Roch

***

In [ ]:
# FOR e-BOOK ONLY
import os, sys
sys.path.insert(0, os.path.abspath('../../utils')) # use directory to mmids.py

In [ ]:
# PYTHON 3
import numpy as np
from numpy import linalg as LA
import matplotlib.pyplot as plt
import pandas as pd
import mmids
seed = 535
rng = np.random.default_rng(seed)
import warnings
warnings.filterwarnings('ignore')

In [ ]:
# FOR TeX ONLY
#plt.rcParams['figure.figsize'] = [4.00, 2.25] # for high-def figs
#plt.rcParams['figure.dpi'] = 200 # for high-def figs

$\newcommand{\bfbeta}{\boldsymbol{\beta}}$ $\newcommand{\bflambda}{\boldsymbol{\lambda}}$

## Application to regression analysis

We return to our motivating example, the regression problem, and apply the least squares approach. 

### Linear and polynomial regression

**Linear regression** We seek an affine function to fit input data points $\{(\mathbf{x}_i, y_i)\}_{i=1}^n$. The common approach involves finding coefficients $\beta_j$'s that minimize the criterion

$$
\sum_{i=1}^n \left(y_i - \left\{\beta_0 + \sum_{j=1}^d \beta_j x_{ij}\right\}\right)^2.
$$

This is indeed a linear least squares problem.

**Figure:** A regression line ([Source](https://commons.wikimedia.org/wiki/File:Linear_least_squares_example2.svg))

![Regression line](https://upload.wikimedia.org/wikipedia/commons/thumb/b/b0/Linear_least_squares_example2.svg/489px-Linear_least_squares_example2.svg.png)

$\bowtie$

<!--TEX
![Regression line ([Source](https://commons.wikimedia.org/wiki/File:Linear_least_squares_example2.svg))](./figs/Linear_least_squares_example2.svg.png)
-->

In matrix form, let

$$
\mathbf{y} = 
\begin{pmatrix}
y_1 \\
y_2 \\
\vdots \\
y_n
\end{pmatrix},
\quad\quad
A =
\begin{pmatrix}
1 & \mathbf{x}_1^T \\
1 & \mathbf{x}_2^T \\
\vdots & \vdots \\
1 & \mathbf{x}_n^T
\end{pmatrix}
\quad\text{and}\quad
\boldsymbol{\beta} = 
\begin{pmatrix}
\beta_0 \\
\beta_1 \\
\vdots \\
\beta_d
\end{pmatrix}.
$$

Then the problem is

$$
\min_{\boldsymbol{\beta}} 
\|\mathbf{y} 
- A \boldsymbol{\beta}\|^2.
$$

We assume that the columns of $A$ are linearly independent, which is typically the case with real data. The normal equations are then

$$
A^T A \boldsymbol{\beta} = A^T \mathbf{y}.
$$

**NUMERICAL CORNER:** We test our least-squares method on simulated data. This has the advantage that we know the truth.

Suppose the truth is a linear function of one variable.

In [ ]:
n, b0, b1 = 100, -1, 1
x = np.linspace(0,10,num=n)
y = b0 + b1*x

In [ ]:
plt.scatter(x,y,alpha=0.5)
plt.show()

A perfect straight line is little too easy. So let's add some noise. That is, to each $y_i$ we add an independent random variable $\varepsilon_i$ with a standard Normal distribution (mean $0$, variance $1$).

In [ ]:
y += rng.normal(0,1,n)

In [ ]:
plt.scatter(x,y,alpha=0.5)
plt.show()

We form the matrix $A$ and use our least-squares code to solve for $\boldsymbol{\hat\beta}$. The function `ls_by_qr`, which we implemented previously, is in [mmids.py](https://raw.githubusercontent.com/MMiDS-textbook/MMiDS-textbook.github.io/main/utils/mmids.py), which is available on the [GitHub of the notes](https://github.com/MMiDS-textbook/MMiDS-textbook.github.io/tree/main). 

In [ ]:
A = np.stack((np.ones(n),x),axis=-1)
coeff = mmids.ls_by_qr(A,y)
print(coeff)

In [ ]:
plt.scatter(x,y,alpha=0.5)
plt.plot(x,coeff[0]+coeff[1]*x,'r')
plt.show()

$\unlhd$

**Beyond linearity** The linear assumption is not as restrictive as it may first appear. The same approach can be extended straightforwardly to fit polynomials or more complicated combination of functions. For instance, suppose $d=1$. To fit a second degree polynomial to the data $\{(x_i, y_i)\}_{i=1}^n$, we add a column to the $A$ matrix with the squares of the $x_i$'s. That is, we let

$$
A =
\begin{pmatrix}
1 & x_1 & x_1^2 \\
1 & x_2 & x_2^2 \\
\vdots & \vdots & \vdots \\
1 & x_n & x_n^2
\end{pmatrix}.
$$

Then, we are indeed fitting a degree-two polynomial as follows 

$$
(A \boldsymbol{\beta})_i 
= \beta_0 + \beta_1 x_i + \beta_2 x_i^2.
$$

The solution otherwise remains the same. 

This idea of adding columns can also be used to model interactions between predictors. Suppose $d=2$. Then we can consider the following $A$ matrix, where the last column combines both predictors into their product,

$$
A =
\begin{pmatrix}
1 & x_{11} & x_{12} & x_{11} x_{12} \\
1 & x_{21} & x_{22} & x_{21} x_{22} \\
\vdots & \vdots & \vdots & \vdots\\
1 & x_{n1} & x_{n2} & x_{n1} x_{n2}
\end{pmatrix}.
$$

**NUMERICAL CORNER:** Suppose the truth is in fact a degree-two polynomial of one variable with Gaussian noise.

In [ ]:
n, b0, b1, b2 = 100, 0, 0, 1
x = np.linspace(0,10,num=n)
y = b0 + b1 * x + b2 * x**2 + 10*rng.normal(0,1,n)

In [ ]:
plt.scatter(x,y,alpha=0.5)
plt.show()

We form the matrix $A$ and use our least-squares code to solve for $\boldsymbol{\hat\beta}$. 

In [ ]:
A = np.stack((np.ones(n), x, x**2), axis=-1)
coeff = mmids.ls_by_qr(A,y)
print(coeff)

In [ ]:
plt.scatter(x,y,alpha=0.5)
plt.plot(x, coeff[0] + coeff[1] * x + coeff[2] * x**2, 'r')
plt.show()

$\unlhd$

### Overfitting in polynomial regression

We return to the `Advertising` dataset from the [[ISLP]](https://www.statlearning.com/) textbook.

**Figure:** Pie chart (*Credit:* Made with [Midjourney](https://www.midjourney.com/))

![Predicting sales](./figs/pie_chart_on_a_screen-small.png)

$\bowtie$

In [ ]:
df = pd.read_csv('advertising.csv')
df.head()

We will focus for now on the TV budget.

In [ ]:
TV = df['TV'].to_numpy()
sales = df['sales'].to_numpy()

In [ ]:
plt.scatter(TV, sales)
plt.xlabel('TV')
plt.ylabel('sales')
plt.show()

We form the matrix $A$ and use our least-squares code to solve for $\boldsymbol{\beta}$. 

In [ ]:
n = np.size(TV)
A = np.stack((np.ones(n),TV),axis=-1)
coeff = mmids.ls_by_qr(A,sales)
print(coeff)

In [ ]:
TVgrid = np.linspace(TV.min(), TV.max(), num=100)
plt.scatter(TV,sales,alpha=0.5)
plt.plot(TVgrid,coeff[0]+coeff[1]*TVgrid,'r')
plt.show()

A degree-two polynomial might be a better fit.

In [ ]:
A = np.stack((np.ones(n), TV, TV**2), axis=-1)
coeff = mmids.ls_by_qr(A,sales)
print(coeff)

In [ ]:
plt.scatter(TV,sales,alpha=0.5)
plt.plot(TVgrid, coeff[0] + coeff[1] * TVgrid + coeff[2] * TVgrid**2,'r')
plt.show()

The fit looks slightly better than the linear one. This is not entirely surprising though given that the linear model is a subset of the quadratic one. But, in adding more parameters, one must worry about [overfitting](https://en.wikipedia.org/wiki/Overfitting#cite_note-1). To quote Wikipedia:

>In statistics, overfitting is "the production of an analysis that corresponds too closely or exactly to a particular set of data, and may therefore fail to fit additional data or predict future observations reliably".[[1](https://en.wikipedia.org/wiki/Overfitting#cite_note-1)] An overfitted model is a statistical model that contains more parameters than can be justified by the data.[[2](https://en.wikipedia.org/wiki/Overfitting#cite_note-CDS-2)] The essence of overfitting is to have unknowingly extracted some of the residual variation (i.e. the noise) as if that variation represented underlying model structure.[[3](https://en.wikipedia.org/wiki/Overfitting#cite_note-BA2002-3)]

To illustrate, let's see what happens with a degree-$20$ polynomial fit.

In [ ]:
deg = 20
A = np.stack([TV**i for i in range(deg+1)], axis=-1)
coeff = mmids.ls_by_qr(A,sales)
print(coeff)

In [ ]:
saleshat = np.sum([coeff[i] * TVgrid**i for i in range(deg+1)], axis=0)

In [ ]:
plt.scatter(TV,sales,alpha=0.5)
plt.plot(TVgrid, saleshat, 'r')
plt.show()

We could use [cross-validation](https://www.textbook.ds100.org/ch/15/bias_cv.html) to choose a suitable degree.

### Bootstrapping for linear regression

We return to the linear case, but with the full set of predictors.

In [ ]:
radio = df['radio'].to_numpy()
newspaper = df['newspaper'].to_numpy()

In [ ]:
f, (ax1, ax2) = plt.subplots(1, 2, sharex=False, sharey=True)
ax1.scatter(radio,sales,alpha=0.5)
ax1.set_title('radio')
ax2.scatter(newspaper,sales,alpha=0.5)
ax2.set_title('newspaper')
plt.show()

In [ ]:
A = np.stack((np.ones(n), TV, radio, newspaper), axis=-1)
coeff = mmids.ls_by_qr(A,sales)
print(coeff)

Newspaper advertising (the last coefficient) seems to have a much weaker effect on sales per dollar spent. Next, we briefly sketch one way to assess the [statistical significance](https://en.wikipedia.org/wiki/Statistical_significance) of such a conclusion.

Our coefficients are estimated from a sample. There is intrinsic variability in our sampling procedure. We would like to understand how our estimated coefficients compare to the true coefficients. This is set up beautifully in [[Data8](https://www.inferentialthinking.com/chapters/13/2/Bootstrap.html), Section 13.2]:

> A data scientist is using the data in a random sample to estimate an unknown parameter. She uses the sample to calculate the value of a statistic that she will use as her estimate. Once she has calculated the observed value of her statistic, she could just present it as her estimate and go on her merry way. But she's a data scientist. She knows that her random sample is just one of numerous possible random samples, and thus her estimate is just one of numerous plausible estimates. By how much could those estimates vary? To answer this, it appears as though she needs to draw another sample from the population, and compute a new estimate based on the new sample. But she doesn't have the resources to go back to the population and draw another sample. It looks as though the data scientist is stuck. Fortunately, a brilliant idea called *the bootstrap* can help her out. Since it is not feasible to generate new samples from the population, the bootstrap generates new random samples by a method called *resampling*: the new samples are drawn at random *from the original sample*.

Without going into full details (see [[DS100](http://www.textbook.ds100.org/ch/17/inf_pred_gen_boot.html), Section 17.3] for more), it works as follows. Let $\{(\mathbf{x}_i, y_i)\}_{i=1}^n$ be our data. We assume that our sample is representative of the population and we simulate our sampling procedure by resampling from the sample. That is, we take a random sample with replacement $\mathcal{X}_{\mathrm{boot},1} = \{(\mathbf{x}_i, y_i)\,:\,i \in I\}$ where $I$ is a [multi-set](https://en.wikipedia.org/wiki/Multiset) of elements from $[n]$ of size $n$. We recompute our estimated coefficients on $\mathcal{X}_{\mathrm{boot},1}$. Then we repeat independently for a desired number of replicates $\mathcal{X}_{\mathrm{boot},1}, \ldots, \mathcal{X}_{\mathrm{boot},r}$. Plotting a histogram of the resulting coefficients gives some idea of the variability of our estimates.

We implement the bootstrap for linear regression in Python next.

In [ ]:
def linregboot(A, b, replicates = np.int32(10000)):
    n,m = A.shape
    coeff_boot = np.zeros((m,replicates))
    for i in range(replicates):
        resample = rng.integers(0,n,n)
        Aboot = A[resample,:]
        bboot = b[resample]
        coeff_boot[:,i] = mmids.ls_by_qr(Aboot,bboot)
    return coeff_boot

First, let's use a simple example from the lecture with a known ground truth. 

In [ ]:
n, b0, b1 = 100, -1, 1
x = np.linspace(0,10,num=n)
y = b0 + b1*x + rng.normal(0,1,n)
A = np.stack((np.ones(n),x),axis=-1)

The estimated coefficients are the following.

In [ ]:
coeff = mmids.ls_by_qr(A,y)
print(coeff)

Now we apply the bootstrap and plot histograms of the two coefficients.

In [ ]:
coeff_boot = linregboot(A,y)

In [ ]:
plt.hist(coeff_boot[0,:])
plt.show()

In [ ]:
plt.hist(coeff_boot[1,:])
plt.show()

We see in the histograms that the true coefficient values $-1$ and $1$ fall within the likely range. 

We return to the `Advertising` dataset and apply the bootstrap.

In [ ]:
n = np.size(TV)
A = np.stack((np.ones(n), TV, radio, newspaper), axis=-1)
coeff = mmids.ls_by_qr(A,sales)
print(coeff)

In [ ]:
coeff_boot = linregboot(A,sales)

Plotting a histogram of the coefficients corresponding to newspaper advertising shows that $0$ is a plausible value, while it is not for TV advertising.

In [ ]:
plt.hist(coeff_boot[1,:])
plt.show()

In [ ]:
plt.hist(coeff_boot[3,:])
plt.show()